先在.env里面配置langsmith的信息，然后在这里写代码运行，然后去langsmith的WEBUI里面看结果

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()
api_key=os.getenv("DEEPSEEK_API_KEY")

llm=init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=api_key,
    temperature=1,  #temperature越低。输出越稳定，0~2，2的话：适合写诗开放性
    # timeout=  ,  超时多少秒未响应，会被取消
    #max_retries= ,  请求失败时候的最大重试次数
)

message=[
    {"role":"system","content":"你是一名数学老师"},
    {"role":"user","content":"1+1=？"},
    {"role":"assistant","content":"我刚刚问了什么问题"}
]

messages2 = [
    SystemMessage(content="你是一个助手"),
    HumanMessage(content="你好")
]

#文本
# info_1=llm.invoke("介绍一下你自己,一句话")
#print(info_1.content)

#字典
# info_2=llm.invoke(message)
# print(info_2.content)

#消息对象
print(llm.invoke(messages2).content)

你好！很高兴见到你，有什么我可以帮你的吗？


练习：


In [4]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel,Field
load_dotenv()
model=init_chat_model(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek:deepseek-chat",
    temperature=0.5,
    configurable_fields=("model","temperature")
)

@tool(parse_docstring=True)
def get_weather(city: str) ->str:
    """获取城市的天气信息
    
    支持中国主要城市的天气查询
    
    Args:
        city (str): 城市名称

    Returns:    
        str: 天气信息

    Examples:
        get_weather("上海")
    """

    return f"{city}今天特别热"
    # """获取城市的天气信息



prompt_template=ChatPromptTemplate.from_messages(
    [
        ("system","你是一个查询天气的助手,你的风格是{style}"),
        ("human","{question}"),
    ]
)


messages=prompt_template.partial(style="热情").invoke({"question": "上海今天天气如何？"})


class weather(BaseModel):
    city:str=Field(...,description="城市名称")
    weather:str=Field(...,description="天气信息")

structured_model=model.with_structured_output(weather)
print(structured_model.invoke(messages))










city='上海' weather='晴转多云，气温18-25℃，微风'
